# Yelp + Amazon ranking (50% density)

Set `PAIR` and run the cell.

- `amazon`: downloads the two raw files from the official SNAP source.
- `yelp`: needs `yelp_academic_dataset_business.json` and `yelp_academic_dataset_review.json` in `/content` or anywhere in Google Drive.

The matrices are checked against the dataset table in the paper. If they differ, the run stops and no ranking is produced.

In [ ]:
PAIR = "amazon"  # or "yelp"

import os, sys, subprocess
DIR = "/content/LU-STW-IETE-revision"
REPO = "https://github.com/Atulpatel123123123/LU-STW-IETE-revision.git"
if not os.path.isdir(DIR):
    subprocess.check_call(["git", "clone", REPO, DIR])
else:
    subprocess.check_call(["git", "-C", DIR, "pull"])
if DIR not in sys.path:
    sys.path.insert(0, DIR)
for k in list(sys.modules):
    if k == "src" or k.startswith("src."):
        del sys.modules[k]

if PAIR == "yelp":
    from google.colab import drive
    drive.mount("/content/drive")

from src.build_paper_matrices import build_yelp, build_amazon
from src.baselines_ranking import run_all_ranking

R_s, R_t = build_yelp() if PAIR == "yelp" else build_amazon()
MODELS = ["AF","KNN-CDRS","KNN-CF","MF-combine","MF-mapping","NNDSVD","MF-HNO3","MF-SNO3","CDRS-GCN","LU-STW"]
df = run_all_ranking(R_s, R_t, train_size=0.5, n_splits=5, seed=42, models=MODELS)
df.to_csv(f"/content/ranking_{PAIR}_50.csv", index=False)
print(df.to_string(index=False))
print("BEST NDCG@10:", df.loc[df["NDCG@10"].idxmax(), "Model"])
